In [1]:
import json
import time
import re
from datetime import datetime, timezone
from pathlib import Path
from getpass import getpass

import pandas as pd
from atproto import Client
from atproto_client.models.app.bsky.feed.search_posts import Params as SearchParams

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)
print(f"Saving raw data to: {RAW_DIR}")

Saving raw data to: /Users/shreyu/VSCODE/junk/UoN-Business-Analytics/Analytics-Specializations-and-Applications/BUSI4370_CW2/data/raw


In [2]:
# BSKY_HANDLE = input("Bluesky handle (e.g. yourname.bsky.social): ").strip()
# BSKY_APP_PASSWORD = getpass("Bluesky App Password (xxxx-xxxx-xxxx-xxxx): ")

BSKY_HANDLE = "shreyuu-dev.bsky.social"
BSKY_APP_PASSWORD = "osip-pbxt-xnfz-52kf"
BASE_URL = "https://bsky.social/xrpc"


client = Client(base_url=BASE_URL)
profile = client.login(BSKY_HANDLE, BSKY_APP_PASSWORD)
print(f"Logged in as: {profile.display_name} (@{profile.handle})")

Logged in as:  (@shreyuu-dev.bsky.social)


In [3]:
def extract_post_record(post) -> dict:
    """Flatten a Bluesky post object to a dict suitable for a pandas row."""
    record = post.record
    author = post.author
    return {
        "uri": post.uri,
        "cid": post.cid,
        "author_did": author.did,
        "author_handle": author.handle,
        "author_display_name": getattr(author, "display_name", None),
        "text": getattr(record, "text", ""),
        "created_at": getattr(record, "created_at", None),
        "indexed_at": getattr(post, "indexed_at", None),
        "langs": ",".join(getattr(record, "langs", []) or []),
        "like_count": getattr(post, "like_count", 0) or 0,
        "repost_count": getattr(post, "repost_count", 0) or 0,
        "reply_count": getattr(post, "reply_count", 0) or 0,
        "quote_count": getattr(post, "quote_count", 0) or 0,
        "is_reply": bool(getattr(record, "reply", None)),
        "embed_type": (
            type(post.embed).__name__ if getattr(post, "embed", None) else None
        ),
    }

In [4]:
def paginated_search(query, max_posts=1000, lang="en", sort="latest",
                     sleep_seconds=0.5, verbose=True):
    """Repeatedly call Bluesky's search_posts endpoint until max_posts retrieved
    or no more results are available."""
    posts = []
    cursor = None
    page = 0
    while len(posts) < max_posts:
        params = SearchParams(
            q=query,
            limit=min(100, max_posts - len(posts)),
            cursor=cursor,
            lang=lang,
            sort=sort,
        )
        try:
            response = client.app.bsky.feed.search_posts(params)
        except Exception as e:
            print(f"  [warn] search call failed on page {page}: {e}")
            break

        page_posts = response.posts
        if not page_posts:
            break

        for post in page_posts:
            try:
                posts.append(extract_post_record(post))
            except Exception as e:
                print(f"  [warn] could not extract a post: {e}")

        cursor = response.cursor
        page += 1
        if verbose:
            print(f"  page {page:>3}: total posts so far = {len(posts):>4}")
        if cursor is None:
            break
        time.sleep(sleep_seconds)
    return posts

In [5]:
def fetch_profiles_batch(dids, batch_size=25, sleep_seconds=0.5):
    """Resolve author DIDs to full profile objects (followers, follows, bio)."""
    profiles = []
    unique_dids = list(dict.fromkeys(dids))
    for i in range(0, len(unique_dids), batch_size):
        batch = unique_dids[i : i + batch_size]
        try:
            response = client.app.bsky.actor.get_profiles({"actors": batch})
        except Exception as e:
            print(f"  [warn] profile batch {i // batch_size} failed: {e}")
            continue

        for p in response.profiles:
            profiles.append({
                "did": p.did,
                "handle": p.handle,
                "display_name": getattr(p, "display_name", None),
                "description": getattr(p, "description", None),
                "followers_count": getattr(p, "followers_count", 0) or 0,
                "follows_count": getattr(p, "follows_count", 0) or 0,
                "posts_count": getattr(p, "posts_count", 0) or 0,
                "created_at": getattr(p, "created_at", None),
                "indexed_at": getattr(p, "indexed_at", None),
                "verified": getattr(p, "verification", None) is not None,
            })
        time.sleep(sleep_seconds)
    return profiles

In [6]:
BRAND_QUERIES = {
    "wired": [
        "wired.com",        # domain
        "@wired.com",       # handle
        '"wired magazine"',     # phrase query
        '"on wired"',
        '"via wired"',
        '"wired says"',      # topical compound to capture article-referencing posts
        '"wired reports"',
        '"wired story"',
        '"wired article"',
        '"wired piece"',
    ],
    "theverge": [
        "theverge.com",     # domain
        "@theverge.com",        # handle
        '"the verge"',      # phrase query
        '"verge says"',     # topical compound to capture article-referencing posts
        '"verge reports"',
        '"verge story"',
        '"verge article"',
        '"verge review"',
        '"according to the verge"',
    ],
}

POSTS_PER_QUERY = 500
TARGET_PER_BRAND = 2500   # collecting generously; cleaning will trim

In [7]:
def collect_brand_posts(brand, queries):
    """Run every query for a brand, deduplicate by URI, return as DataFrame."""
    print(f"\n=== Collecting posts for: {brand} ===")
    all_posts = []
    for q in queries:
        print(f"  Query: {q!r}")
        posts = paginated_search(query=q, max_posts=POSTS_PER_QUERY)
        for p in posts:
            p["query"] = q
        all_posts.extend(posts)
        print(f"   -> {len(posts)} posts retrieved for this query")

    df = pd.DataFrame(all_posts)
    if df.empty:
        return df

    before = len(df)
    df = df.drop_duplicates(subset="uri", keep="first").reset_index(drop=True)
    print(f"  Deduplication: {before} -> {len(df)} unique posts")

    if len(df) > TARGET_PER_BRAND:
        df = df.head(TARGET_PER_BRAND).reset_index(drop=True)

    df["brand"] = brand
    return df

In [8]:
wired_df = collect_brand_posts("wired", BRAND_QUERIES["wired"])
print(f"\nWired total: {len(wired_df)} unique posts, {wired_df['author_did'].nunique()} unique authors")
wired_df[["author_handle", "text", "like_count", "repost_count"]].head(5)


=== Collecting posts for: wired ===
  Query: 'wired.com'
  page   1: total posts so far =   99
  page   2: total posts so far =  197
  page   3: total posts so far =  297
  page   4: total posts so far =  394
  page   5: total posts so far =  494
  page   6: total posts so far =  500
   -> 500 posts retrieved for this query
  Query: '@wired.com'
  page   1: total posts so far =   99
  page   2: total posts so far =  197
  page   3: total posts so far =  297
  page   4: total posts so far =  394
  page   5: total posts so far =  492
  page   6: total posts so far =  500
   -> 500 posts retrieved for this query
  Query: '"wired magazine"'
  page   1: total posts so far =   97
  page   2: total posts so far =  195
  page   3: total posts so far =  292
  page   4: total posts so far =  387
  page   5: total posts so far =  484
  page   6: total posts so far =  500
   -> 500 posts retrieved for this query
  Query: '"on wired"'
  page   1: total posts so far =   96
  page   2: total posts s

,author_handle,text,like_count,repost_count
0,longtail-news.bsky.social,"Feed: ""Ars Technica - All content""\nBy: Boone ...",0,0
1,dustandroad.bsky.social,‘It’s Undignified’: Hundreds of Workers Traini...,0,0
2,remalone.bsky.social,Concerned about Trump seizing election records...,2,3
3,toddcorley.bsky.social,It is VERY telling when corporations like @pro...,0,0
4,jeremywired.bsky.social,"GETTING HANDSY: ""A robot’s claw hurtles toward...",1,1


In [9]:
def highlight_wired(val):
    """Visual sanity-check that retrieved posts are actually about Wired."""
    pattern = r'(wired\.com|@wired\.com|wired\s+(magazine|says|reports|story|article|piece))'
    return re.sub(pattern, r'<mark>\1</mark>', str(val), flags=re.IGNORECASE)

wired_df['text'].head(15).to_frame().style.format(highlight_wired)

,text
0,"Feed: ""Ars Technica - All content"" By: Boone Ashworth, wired.com on Wednesday, April 29, 2026"
1,"‘It’s Undignified’: Hundreds of Workers Training Meta’s #AI Could Be Laid Off. More than 700 people working for a #Meta contractor in Ireland are at risk of losing their jobs, documents show. Via @wired.com"
2,Concerned about Trump seizing election records? You should be. Nobody left in the Voting Rights section at DOJ to care. Great work from @wired.com : www.wired.com/story/the-ju...
3,It is VERY telling when corporations like @proton take actions such as this. As if they will get away from all their lies. @wired.com should do a story on how @PROTON is actually compromised and NOT living up to what they promise users!!!
4,"GETTING HANDSY: ""A robot’s claw hurtles toward a light bulb. I wince, waiting for the crunch. The bulb rolls away. After a few nips, the bulb is in its grasp. In 10+ years writing about robots I've never seen one move so naturally."" @willknight.bsky.social on @wired.com www.wired.com/story/when-r..."
5,"Shut up!! Shut up about goblins!! New in @wired.com: 'Never talk about goblins, gremlins, raccoons, trolls, ogres, pigeons, or other animals or creatures unless it is absolutely and unambiguously relevant' www.wired.com/story/openai..."
6,“AI agents may soon be buying your stuff for you. The FIDO Alliance has teamed up with Google and Mastercard to try to ensure that shopping in the near future isn't a complete disaster.” www.wired.com/story/the-ra... @wired.com
7,@eff.org @wired.com @youranoncentral.bsky.social @indivisible.org @teslatakedown.com @housedemocrats.bsky.social @dscc.bsky.social @meidastouch.com @briantylercohen.bsky.social
8,#EpsteinClass @eff.org @wired.com @indivisible.org @teslatakedown.com @youranoncentral.bsky.social @votersoftomorrow.org
9,@wired.com WHAT DO YOU MEAN THIS IS MY LAST FREE ARTICLE I JUST GOT HERE I HAVEN'T BEEN TO YOUR SITE IN MONTHS


In [10]:
verge_df = collect_brand_posts("theverge", BRAND_QUERIES["theverge"])
print(f"\nThe Verge total: {len(verge_df)} unique posts, {verge_df['author_did'].nunique()} unique authors")
verge_df[["author_handle", "text", "like_count", "repost_count"]].head(5)


=== Collecting posts for: theverge ===
  Query: 'theverge.com'
  page   1: total posts so far =   99
  page   2: total posts so far =  198
  page   3: total posts so far =  297
  page   4: total posts so far =  395
  page   5: total posts so far =  491
  page   6: total posts so far =  498
  page   7: total posts so far =  500
   -> 500 posts retrieved for this query
  Query: '@theverge.com'
  page   1: total posts so far =   98
  page   2: total posts so far =  197
  page   3: total posts so far =  296
  page   4: total posts so far =  391
  page   5: total posts so far =  488
  page   6: total posts so far =  500
   -> 500 posts retrieved for this query
  Query: '"the verge"'
  page   1: total posts so far =  100
  page   2: total posts so far =  199
  page   3: total posts so far =  298
  page   4: total posts so far =  398
  page   5: total posts so far =  496
  page   6: total posts so far =  500
   -> 500 posts retrieved for this query
  Query: '"verge says"'
  page   1: total p

,author_handle,text,like_count,repost_count
0,kcurry.bsky.social,google search queries hit an all-time high in ...,0,0
1,heflin.bsky.social,I really need to free up some subscription mon...,0,0
2,kcurry.bsky.social,xbox email addresses coming for xbox employees...,0,0
3,idlesloth.bsky.social,Mojang employees will also be given an mojangc...,1,0
4,forensicgarlic.bsky.social,I already know what the 'Brendan Carr is a dum...,0,0


In [11]:
def highlight_verge(val):
    pattern = r'(theverge\.com|@theverge\.com|the\s+verge|verge\s+(says|reports|story|article|review))'
    return re.sub(pattern, r'<mark>\1</mark>', str(val), flags=re.IGNORECASE)

verge_df['text'].head(15).to_frame().style.format(highlight_verge)

,text
0,google search queries hit an all-time high in q1 2026 - probably has something to do with all the ai chatbots we're using these days. {https://theverge.com} https://www.theverge.com/tech/920815/google-alphabet-q1-2026-earnings-sundar-pichai
1,I really need to free up some subscription money for @theverge.com. @lopatto.bsky.social's reporting alone would be worth it.
2,"xbox email addresses coming for xbox employees - because ""we are xbox"" now apparently. {https://theverge.com/rss/index.xml} https://www.theverge.com/report/920525/microsoft-xbox-email-address-change"
3,"Mojang employees will also be given an mojangcom email address, and both Xbox and Mojang employees will still retain their microsoftcom email aliases. The change is being made as part of “strengthening the Xbox identity inside and outside of Microsoft,” theverge.com/report/92052..."
4,I already know what the 'Brendan Carr is a dummy ' segment of the @theverge.com podcast is going to be about...
5,@joncooper-us.bsky.social @andrewjweinstein.com @propublica.org @theverge.com @cookpolitical.com @urocklive1.bsky.social @bluestormcomin1.bsky.social @cwebbonline.com @teapainusa.bsky.social @meidastouch.com @georgetakei.bsky.social @gtconway.bsky.social @aclu.org @patrioticmillionaires.org
6,"@theverge.com do a blog on arena sideline tech: clocks, scoreboards, buzzers"
7,Is it time for America's favourite podcast within a podcast???? @theverge.com
8,elon musk testifies in trial against openai co-founders. this is getting real. {theverge.com} https://www.theverge.com/ai-artificial-intelligence/917052/elon-musk-takes-stand-trial-openai-sam-altman
9,"Can't wait until this week's episode of ""Brendan Carr is a Dummy"" @reckless.bsky.social @davidpierce.xyz @theverge.com"


In [12]:
wired_path = RAW_DIR / "wired_posts_raw.csv"
verge_path = RAW_DIR / "theverge_posts_raw.csv"

wired_df.to_csv(wired_path, index=False)
verge_df.to_csv(verge_path, index=False)

print(f"Saved {len(wired_df)} Wired posts -> {wired_path}")
print(f"Saved {len(verge_df)} Verge posts -> {verge_path}")

Saved 2500 Wired posts -> /Users/shreyu/VSCODE/junk/UoN-Business-Analytics/Analytics-Specializations-and-Applications/BUSI4370_CW2/data/raw/wired_posts_raw.csv
Saved 1946 Verge posts -> /Users/shreyu/VSCODE/junk/UoN-Business-Analytics/Analytics-Specializations-and-Applications/BUSI4370_CW2/data/raw/theverge_posts_raw.csv


In [13]:
print("Fetching profiles for Wired authors...")
wired_dids = wired_df["author_did"].dropna().unique().tolist()
wired_profiles = pd.DataFrame(fetch_profiles_batch(wired_dids))
print(f"  -> {len(wired_profiles)} profiles fetched")

print("\nFetching profiles for The Verge authors...")
verge_dids = verge_df["author_did"].dropna().unique().tolist()
verge_profiles = pd.DataFrame(fetch_profiles_batch(verge_dids))
print(f"  -> {len(verge_profiles)} profiles fetched")

Fetching profiles for Wired authors...
  -> 1597 profiles fetched

Fetching profiles for The Verge authors...
  -> 1527 profiles fetched


In [14]:
wired_profiles_path = RAW_DIR / "wired_profiles_raw.csv"
verge_profiles_path = RAW_DIR / "theverge_profiles_raw.csv"

wired_profiles.to_csv(wired_profiles_path, index=False)
verge_profiles.to_csv(verge_profiles_path, index=False)

print(f"Saved {len(wired_profiles)} Wired profiles")
print(f"Saved {len(verge_profiles)} Verge profiles")

Saved 1597 Wired profiles
Saved 1527 Verge profiles


In [15]:
def date_range(df):
    if df.empty or "created_at" not in df.columns:
        return ("n/a", "n/a")
    s = pd.to_datetime(df["created_at"], errors="coerce", utc=True)
    return (str(s.min()), str(s.max()))

metadata = {
    "collection_run_at": datetime.now(timezone.utc).isoformat(),
    "collected_by_handle": BSKY_HANDLE,
    "language_filter": "en",
    "sort_order": "latest",
    "geographic_scope": "global (Bluesky search does not expose location filtering)",
    "brand_pair_rationale": (
        "Wired vs The Verge selected as direct competitors in technology "
        "journalism with distinct editorial positioning (long-form/culture vs "
        "product-review/news-cycle). Brand pair was changed from an initial "
        "Hugging Face vs Replicate selection after pilot data showed that "
        "'replicate' as a common English verb produced unresolvable lexical "
        "collision under any reasonable precision/recall trade-off."
    ),
    "brands": {
        "wired": {
            "queries": BRAND_QUERIES["wired"],
            "n_posts": int(len(wired_df)),
            "n_unique_users": int(wired_df["author_did"].nunique()) if not wired_df.empty else 0,
            "date_range": date_range(wired_df),
            "n_profiles_fetched": int(len(wired_profiles)),
        },
        "theverge": {
            "queries": BRAND_QUERIES["theverge"],
            "n_posts": int(len(verge_df)),
            "n_unique_users": int(verge_df["author_did"].nunique()) if not verge_df.empty else 0,
            "date_range": date_range(verge_df),
            "n_profiles_fetched": int(len(verge_profiles)),
        },
    },
}

meta_path = RAW_DIR / "collection_metadata.json"
with open(meta_path, "w") as f:
    json.dump(metadata, f, indent=2)

print(json.dumps(metadata, indent=2))

{
  "collection_run_at": "2026-04-30T07:52:31.646759+00:00",
  "collected_by_handle": "shreyuu-dev.bsky.social",
  "language_filter": "en",
  "sort_order": "latest",
  "geographic_scope": "global (Bluesky search does not expose location filtering)",
  "brand_pair_rationale": "Wired vs The Verge selected as direct competitors in technology journalism with distinct editorial positioning (long-form/culture vs product-review/news-cycle). Brand pair was changed from an initial Hugging Face vs Replicate selection after pilot data showed that 'replicate' as a common English verb produced unresolvable lexical collision under any reasonable precision/recall trade-off.",
  "brands": {
    "wired": {
      "queries": [
        "wired.com",
        "@wired.com",
        "\"wired magazine\"",
        "\"on wired\"",
        "\"via wired\"",
        "\"wired says\"",
        "\"wired reports\"",
        "\"wired story\"",
        "\"wired article\"",
        "\"wired piece\""
      ],
      "n_posts

In [16]:
def quick_summary(df, name):
    print(f"--- {name} ---")
    if df.empty:
        print(" (empty)")
        return
    print(f"  Total posts:     {len(df)}")
    print(f"  Unique authors:  {df['author_did'].nunique()}")
    print(f"  Date range:      {date_range(df)}")
    print(f"  Mean likes/post: {df['like_count'].mean():.2f}")
    print(f"  Mean reposts/post: {df['repost_count'].mean():.2f}")
    print(f"  Reply share:     {df['is_reply'].mean():.1%}")

quick_summary(wired_df, "Wired")
print()
quick_summary(verge_df, "The Verge")

--- Wired ---
  Total posts:     2500
  Unique authors:  1597
  Date range:      ('2023-09-21 16:45:42.951000+00:00', '2026-04-30 02:28:41.355146+00:00')
  Mean likes/post: 18.93
  Mean reposts/post: 5.60
  Reply share:     29.1%

--- The Verge ---
  Total posts:     1946
  Unique authors:  1527
  Date range:      ('2023-07-03 17:39:11.855000+00:00', '2026-04-30 07:28:13.237000+00:00')
  Mean likes/post: 11.99
  Mean reposts/post: 2.83
  Reply share:     35.7%
